# INSTRUCTOR SOLUTION: Building a Validated REST API for ML Model Serving
## AIAT 125 — Unit 2: Packaging and Serving AI Models

**⚠️ INSTRUCTOR USE ONLY — Do not distribute to students**

| Task | Points |
|---|---|
| Task 1: `/predict` endpoint with validation | 30 |
| Task 2: Automated test suite (4 cases) | 25 |
| Task 3: Latency benchmark p50/p95 | 20 |
| Task 4: `/health` endpoint with 3 required fields | 25 |

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "flask", "scikit-learn", "joblib"], check=False)

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib, numpy as np, json, time, os

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)

MODEL_PATH = "/tmp/iris_rf_exercise.joblib"
joblib.dump(clf, MODEL_PATH)
print(f"Model saved: accuracy = {clf.score(X_test, y_test):.2%}")
print("Setup complete.")

---
## Task 1 — `/predict` Endpoint with Input Validation (30 points)

In [ ]:
from flask import Flask, request, jsonify
import joblib, numpy as np

app = Flask(__name__)

CLASS_NAMES = ["setosa", "versicolor", "virginica"]
REQUIRED_KEYS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
_model = joblib.load(MODEL_PATH)
_app_version = "1.0.0"


@app.route("/health")
def health():
    # SOLUTION Task 4: return all 3 required fields
    return jsonify({"status": "ok", "model_loaded": True, "version": _app_version})


@app.route("/predict", methods=["POST"])
def predict():
    # SOLUTION 1a: Parse JSON body
    data = request.get_json(silent=True)
    if data is None or not isinstance(data, dict):
        return jsonify({"error": "Request body must be a JSON object"}), 400

    # SOLUTION 1b: Check all required keys present
    missing = [k for k in REQUIRED_KEYS if k not in data]
    if missing:
        return jsonify({"error": f"Missing fields: {missing}"}), 422

    # SOLUTION 1c: Convert values to float
    try:
        features = [float(data[k]) for k in REQUIRED_KEYS]
    except (ValueError, TypeError):
        return jsonify({"error": "All field values must be numeric"}), 422

    # SOLUTION 1d: Run prediction
    arr = np.array([features])
    proba = _model.predict_proba(arr)[0]
    class_idx = int(np.argmax(proba))
    return jsonify({
        "prediction": CLASS_NAMES[class_idx],
        "confidence": round(float(proba[class_idx]), 4)
    })


print("Flask app defined.")

---
## Task 2 — Automated Test Suite (25 points)

In [ ]:
client = app.test_client()

# --- Test A: Happy path ---
valid_payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
# SOLUTION: send POST with valid_payload
resp_a = client.post("/predict",
                     data=json.dumps(valid_payload),
                     content_type="application/json")
body_a = json.loads(resp_a.data)
test_a_passed = resp_a.status_code == 200 and "prediction" in body_a and "confidence" in body_a

# --- Test B: Missing field ---
partial_payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4}  # no petal_width
# SOLUTION
resp_b = client.post("/predict",
                     data=json.dumps(partial_payload),
                     content_type="application/json")
test_b_passed = resp_b.status_code == 422

# --- Test C: Wrong type ---
bad_type_payload = {"sepal_length": "five", "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
# SOLUTION
resp_c = client.post("/predict",
                     data=json.dumps(bad_type_payload),
                     content_type="application/json")
test_c_passed = resp_c.status_code == 422

# --- Test D: Empty body ---
# SOLUTION
resp_d = client.post("/predict", data="", content_type="application/json")
test_d_passed = resp_d.status_code == 400

# --- Print results ---
results = {
    "A: happy path (expect 200)"  : test_a_passed,
    "B: missing field (expect 422)": test_b_passed,
    "C: wrong type (expect 422)"  : test_c_passed,
    "D: empty body (expect 400)"  : test_d_passed,
}
print("=== API Test Suite ===")
all_passed = True
for name, passed in results.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}")
    if not passed:
        all_passed = False

assert all_passed, "Some tests failed"
print("\nTask 2 PASSED")

---
## Task 3 — Latency Benchmark (20 points)

In [ ]:
payload_str = json.dumps({"sepal_length": 5.1, "sepal_width": 3.5,
                           "petal_length": 1.4, "petal_width": 0.2})

# SOLUTION: Run 50 POST requests and measure each
bench_latencies = []
for _ in range(50):
    t0 = time.perf_counter()
    client.post("/predict", data=payload_str, content_type="application/json")
    t1 = time.perf_counter()
    bench_latencies.append((t1 - t0) * 1000.0)

# SOLUTION: Compute p50 and p95
import numpy as np
bench_p50 = np.percentile(bench_latencies, 50)
bench_p95 = np.percentile(bench_latencies, 95)

print(f"Benchmark (50 requests):")
print(f"  p50 (median) : {bench_p50:.2f} ms")
print(f"  p95          : {bench_p95:.2f} ms")

assert len(bench_latencies) == 50, f"Need 50 latencies, got {len(bench_latencies)}"
assert bench_p50 is not None and bench_p95 is not None
assert bench_p95 < 500, f"p95 should be < 500ms in-process, got {bench_p95:.1f}ms"
print("Task 3 PASSED")

---
## Task 4 — `/health` Endpoint (25 points)

The `/health` endpoint was already implemented in Task 1 (returning `status`, `model_loaded`, `version`).

In [ ]:
# SOLUTION: Test /health
resp_health = client.get("/health")
body_health = json.loads(resp_health.data)

print(f"Health status code : {resp_health.status_code}")
print(f"Health body        : {body_health}")

# SOLUTION: 3 assertions
assert resp_health.status_code == 200, f"Expected 200, got {resp_health.status_code}"
assert body_health["status"] == "ok", f"Expected 'ok', got {body_health['status']}"
assert body_health["model_loaded"] == True, "model_loaded must be True"

print("Task 4 PASSED")

In [ ]:
# --- Final summary ---
print("=" * 55)
print("UNIT 2 LAB — FINAL DEPLOYMENT GATE")
print("=" * 55)

checks = {
    "Task 1: /predict returns 200 on valid input" :
        resp_a is not None and resp_a.status_code == 200,
    "Task 2: all 4 test cases pass" : all_passed,
    "Task 3: 50-request benchmark p95 < 500ms" :
        bench_p95 is not None and bench_p95 < 500,
    "Task 4: /health has status/model_loaded/version" :
        resp_health.status_code == 200 and
        all(k in body_health for k in ["status", "model_loaded", "version"]),
}
for task, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {task}")
print("=" * 55)